# Simonyi Mount Motion

This notebook goes through the process of investigating exposures that were affected by considerable mount motion with the goal of identifying such exposures and adding them to the [excluded_visits](https://github.com/lsst-dm/excluded_visits).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.io.fits as pf
from astropy.time import Time, TimeDelta
from astropy.table import Table
import pandas as pd
import os
os.environ["no_proxy"] += ",.consdb"

In [ ]:
from lsst.summit.utils import ConsDbClient
from lsst.summit.utils.utils import computeCcdExposureId

client = ConsDbClient('http://consdb-pq.consdb:8080/consdb')
#schema = 'lsstcomcam'
schema = 'lsstcam'
table = 'ccdvisit1_quicklook'
table = 'exposure_quicklook'
table = 'visit1_quicklook'
print(f'schemas:\n', client.schema())  # list the instruments
print(f'tables (schema={schema}):\n', client.schema(schema))  # list tables for an instrument
print(f'columns (table={table}): {table}\n', sorted(list(client.schema(schema, f'cdb_{schema}.{table}').keys())))

In [ ]:
from lsst.summit.utils import ConsDbClient
from lsst.summit.utils.utils import computeCcdExposureId

client = ConsDbClient('http://consdb-pq.consdb:8080/consdb')

query = """
SELECT q.day_obs, e.exp_midpt_mjd, q.exposure_id, q.seq_num, e.band, e.exp_time, 
s_ra as ra, s_dec as dec, img_type,
v.psf_sigma_median,
q.mount_motion_image_degradation,
q.mount_motion_image_degradation_az, 
q.mount_motion_image_degradation_el,
q.mount_jitter_rms_cam_hexapod, 
q.mount_jitter_rms_m2_hexapod,
q.mount_jitter_rms_rot
FROM cdb_lsstcam.exposure_quicklook q, cdb_lsstcam.exposure e, cdb_lsstcam.visit1_quicklook v
WHERE q.exposure_id = e.exposure_id 
and q.exposure_id = v.visit_id
AND q.day_obs > 20250410 
AND e.img_type = 'science'
order by q.exposure_id
"""
tab = client.query(query)
mount_cols = columns = ['mount_motion_image_degradation','mount_motion_image_degradation_az', 'mount_motion_image_degradation_el']
jitter_cols = ['mount_jitter_rms_cam_hexapod', 'mount_jitter_rms_m2_hexapod', 'mount_jitter_rms_rot']
for col in mount_cols + jitter_cols + ['exp_midpt_mjd']:
    tab[col] = tab[col].astype(float)

df = tab.to_pandas()
df

In [ ]:
# Get the excluded visits from github repo
#url = "https://raw.githubusercontent.com/lsst-dm/excluded_visits/refs/heads/main/LSSTCam/bad.ecsv"
url = "https://raw.githubusercontent.com/lsst-dm/excluded_visits/refs/heads/u/kadrlica/20260610/LSSTCam/bad.ecsv"
table = Table.read(url, format="ascii.ecsv").to_pandas()
trailed = table[table['comment'].str.contains('trailed')]
print(f"Number of excluded visits: {len(table)}")
print(f"Number of trailed visits: {len(trailed)}")

In [ ]:
# Thresholds for identifying "suspicious" exposures
THRESH = {
    'mount_motion_image_degradation': 0.2, 
    'mount_motion_image_degradation_az': 0.04,
    'mount_motion_image_degradation_el': 0.2,
    'mount_jitter_rms_cam_hexapod': 0.1,
    'mount_jitter_rms_m2_hexapod': 0.05,
    'mount_jitter_rms_rot': 1.0,
}

#DATE = 20260528
DATE = 20300101

In [ ]:
sel1 = df['day_obs'] < DATE
sel2 = df['exposure_id'].isin(trailed['exposure'])

kw = dict(bins=np.linspace(0,200,100), log=True)
kw = dict(bins=np.linspace(0,1,100), log=True)
#kw['bins'] = np.logspace(-2,3)

fig,ax = plt.subplots(3,1,figsize=(10,11))
for i,col in enumerate(mount_cols):
    ax[i].hist(df[sel1][col], label=f'Science Exposures (<{DATE})', **kw)
    #ax[i].hist(df[sel][col], label='Exclude List Trailed', **kw)
    ax[i].hist(df[sel1 & sel2][col], label=f'Excluded Exposures (<{DATE})', **kw)
    ax[i].axvline(THRESH[col], ls='--', color='gray', lw=2)
    ax[i].legend()
    ax[i].set_xlabel(col)
    ax[i].set_ylabel("Number of Exposures")

In [ ]:
sel1 = df['day_obs'] < DATE
sel2 = df['exposure_id'].isin(trailed['exposure'])

kw = dict(bins=np.linspace(0,100,100), log=True)
kw = dict(bins=np.linspace(0,1,100), log=True)
#kw['bins'] = np.logspace(-2,3)

fig,ax = plt.subplots(3,1,figsize=(10,11))
for i,col in enumerate(jitter_cols):
    ax[i].hist(df[sel1][col], label=f'Science Exposures (< {DATE})', **kw)
    ax[i].hist(df[sel1 & sel2][col], label=f'Trailed Exposures (< {DATE})', **kw)
    ax[i].axvline(THRESH[col], ls='--', color='gray', lw=2)
    ax[i].legend()
    ax[i].set_xlabel(col)
    ax[i].set_ylabel("Number of Exposures")

In [ ]:
#sel1 = np.ones(len(df),dtype=bool) 
sel1 = df['day_obs'] < DATE
sel2 = df['exposure_id'].isin(trailed['exposure'])

kw = dict(alpha=0.5, s=5)

fig,ax = plt.subplots(3,1,figsize=(10,11))
for i,col in enumerate(mount_cols):
    ax[i].scatter(df[sel1]['exp_midpt_mjd'], df[sel1][col], label=f'Science Exposures (< {DATE})', **kw)
    ax[i].scatter(df[sel1 & sel2]['exp_midpt_mjd'], df[sel1 & sel2][col], label=f'Trailed Exposures (< {DATE})', **kw)
    ax[i].axhline(THRESH[col], ls='--', color='gray', lw=2)
    ax[i].set_ylabel(col)
    ax[i].set_yscale('log')
ax[0].legend()
ax[-1].set_xlabel('exp_midpt_mjd')

fig,ax = plt.subplots(3,1,figsize=(10,11))
for i,col in enumerate(jitter_cols):
    ax[i].scatter(df[sel1]['exp_midpt_mjd'], df[sel1][col], label=f'Science Exposures (< {DATE})', **kw)
    ax[i].scatter(df[sel1 & sel2]['exp_midpt_mjd'], df[sel1 & sel2][col], label=f'Trailed Exposures (< {DATE})', **kw)
    ax[i].axhline(THRESH[col], ls='--', color='gray', lw=2)
    ax[i].set_ylabel(col)
    ax[i].set_yscale('log')
ax[0].legend()
ax[-1].set_xlabel('exp_midpt_mjd')

In [ ]:
sel1 = df['day_obs'] < 20260528
sel2 = df['exposure_id'].isin(trailed['exposure'])

psf_col = 'psf_sigma_median'

kw = dict(alpha=0.5, s=5)
fig,ax = plt.subplots(1,3,figsize=(18,5))
for i,col in enumerate(mount_cols):
    ax[i].scatter( df[sel1][col], df[sel1][psf_col], label='Science Exposures (< 20260528)', **kw)
    ax[i].scatter( df[sel1 & sel2][col], df[sel1 & sel2][psf_col], label='Trailed Exposures (< 20260528)', **kw)
    ax[i].set_ylim(0,10)
    ax[i].set_xscale('log')
    ax[i].set_xlabel(col)
ax[0].set_ylabel(psf_col)
ax[-1].legend()

fig,ax = plt.subplots(1,3,figsize=(18,5))
for i,col in enumerate(jitter_cols):
    ax[i].scatter( df[sel1][col], df[sel1][psf_col], label='Science Exposures (< 20260528)', **kw)
    ax[i].scatter( df[sel1 & sel2][col], df[sel1 & sel2][psf_col], label='Trailed Exposures (< 20260528)', **kw)
    ax[i].set_ylim(0,10)
    ax[i].set_xscale('log')
    ax[i].set_xlabel(col)
ax[0].set_ylabel(psf_col)
ax[-1].legend()

In [ ]:
print(f"Number of excluded exposures: {len(table)}")
print(f"Number of excluded exposures in consdb query: {np.isin(table['exposure'], df['exposure_id']).sum()}")
print(f"Number of consdb exposures: {len(df)}")
print(f"Number of consdb exposures in exclude list: {np.isin(df['exposure_id'], table['exposure']).sum()}")

In [ ]:
sel = df['day_obs'] < DATE
sel &= df['exposure_id'].isin(trailed['exposure'])

print(f"Trailed exposures & (day_obs < {DATE}): {sel.sum()}")
for col in mount_cols:
    sel &= df[col] < THRESH[col]
    print(f"& {col} < {THRESH[col]}: {sel.sum()}")
for col in jitter_cols:
    sel &= df[col] < THRESH[col]
    print(f"& {col} < {THRESH[col]}: {sel.sum()}")  

In [ ]:
# Select images to investigate

"""
sel = df['day_obs'] < DATE
sel &= (df['mount_motion_image_degradation'] > 0.2)
#sel &= (df['mount_motion_image_degradation_az'] > 0.04) 
#sel &= (df['mount_motion_image_degradation_el'] > 0.2)
sel &= ~df['exposure_id'].isin(table['exposure'])
"""


"""
# Bad mount jitter rot
sel = df['day_obs'] < 20260528
sel &= ~df['exposure_id'].isin(table['exposure'])
#sel &= (df['mount_jitter_rms_rot'] > 1) 
sel &= (df['mount_jitter_rms_cam_hexapod'] > 0.2) & (df['mount_jitter_rms_cam_hexapod'] < 1)
#sel &= (df['mount_jitter_rms_m2_hexapod'] > 0.1) & (df['mount_jitter_rms_m2_hexapod'] < 1)
"""

sel = df['day_obs'] < DATE
sel &= ~(df['exposure_id'].isin(table['exposure']))
col_sel = np.zeros(len(df),dtype=bool)
#cols = ['mount_jitter_rms_rot']
cols = mount_cols + jitter_cols
for col in cols:
    print(f"{col} > {THRESH[col]}")
    col_sel |= df[col] > THRESH[col]
sel &= col_sel

"""
# Exposures that are marked as trailed by don't fail any mount motion or jitter test
sel  = df['day_obs'] < DATE
sel &= df['exposure_id'].isin(trailed['exposure'])
for col in mount_cols + jitter_cols:
    sel &= df[col] < THRESH[col]
"""

print(f"Selected exposures: {sel.sum()}")

In [ ]:
def link_to_witness(df):
    baseurl = "https://usdf-rsp-dev.slac.stanford.edu/rubintv/summit-usdf/lsstcam/"
    
    for idx, row in df.iterrows():
        exposure_id = str(row['exposure_id'])
        year = int(exposure_id[:4])
        month = int(exposure_id[4:6])
        day = int(exposure_id[6:8])
        seq_num = int(row['seq_num'])
        
        query = f"event?channel_name=witness_detector&date_str={year}-{month:02d}-{day:02d}&seq_num={seq_num}&ext=jpg"
        url = baseurl + query
        msg = f"{row['exposure_id']}: {url}\n"
        for col in mount_cols:
            msg += f"  {col}={row[col]:.2f}"
        msg += '\n'
        for col in jitter_cols:
            msg += f"  {col}={row[col]:.2f}"
        print(msg)

link_to_witness(df[sel])

In [ ]:
# These look ok....
#https://usdf-rsp-dev.slac.stanford.edu/rubintv/summit-usdf/lsstcam/event?channel_name=witness_detector&date_str=2026-05-28&seq_num=513&ext=jpg
#https://usdf-rsp-dev.slac.stanford.edu/rubintv/summit-usdf/lsstcam/event?channel_name=witness_detector&date_str=2026-05-28&seq_num=715&ext=jpg
#https://usdf-rsp-dev.slac.stanford.edu/rubintv/summit-usdf/lsstcam/event?channel_name=witness_detector&date_str=2026-05-28&seq_num=716&ext=jpg
#
# Lauren reports "Azimuth & Elevation plots seem to be missing data for those"
#https://usdf-rsp-dev.slac.stanford.edu/rubintv/summit-usdf/lsstcam/event?channel_name=mount&date_str=2026-05-28&seq_num=513&ext=png

# Looks ok...
# https://usdf-rsp-dev.slac.stanford.edu/rubintv/summit-usdf/lsstcam/event?channel_name=witness_detector&date_str=2026-04-23&seq_num=174&ext=jpg
# https://usdf-rsp-dev.slac.stanford.edu/rubintv/summit-usdf/lsstcam/event?channel_name=witness_detector&date_str=2026-05-29&seq_num=322&ext=jpg
# https://usdf-rsp-dev.slac.stanford.edu/rubintv/summit-usdf/lsstcam/event?channel_name=witness_detector&date_str=2026-05-29&seq_num=663&ext=jpg
# https://usdf-rsp-dev.slac.stanford.edu/rubintv/summit-usdf/lsstcam/event?channel_name=witness_detector&date_str=2026-05-29&seq_num=667&ext=jpg

In [ ]:
# All exposures in DP2
from lsst.daf.butler import Butler

repo = "dp2_prep" 
collection = ['LSSTCam/runs/DRP/DP2'] 
instrument = "LSSTCam"
skymap = 'lsst_cells_v2'

butler = Butler(
        repo,
        collections=collection,
        instrument=instrument)

refs = butler.query_datasets('visit_summary', limit=1000000)
visits = np.unique([r.dataId['visit'] for r in refs])
print(f"Number of all visits: {len(visits)}")

ref = butler.query_datasets('deep_coadd_input_summary')[0]
summary_table = butler.get(ref)
coadd_visits = np.unique(summary_table['visit'])
print(f"Number of coadd visits: {len(coadd_visits)}")

In [ ]:
sel = np.isin(visits, table['exposure'])
print(f"Visits from DP2 visit_summary in excluded list: {sel.sum()}")
print(visits[sel])

sel = np.isin(coadd_visits, table['exposure'])
print(f"Visits from DP2 deep_coadd_input_summary in excluded list: {sel.sum()}")
print(coadd_visits[sel])
    
link_to_witness(df[np.isin(df['exposure_id'], coadd_visits[sel])])

In [ ]:
expid = 2025050600762
expid = 2025071100501
print(summary_table[summary_table['visit'] == expid])
df[df['exposure_id'] == expid][['band','ra','dec']]

In [ ]:
np.unique(summary_table[ (summary_table['visit'] == expid) & (summary_table['tract'] == 3534)]['patch'])

In [ ]:
from lsst.daf.butler import Butler
import lsst.afw.display as afwDisplay

tract, patch, band = 6068, 19, 'y'
dataId={"tract": tract, "patch": patch, "band": band, "skymap": skymap}
butler = Butler(repo, collections=collection)
coadd = butler.get( "deep_coadd", dataId)

In [ ]:
from lsst.afw.fits import CompressionOptions, CompressionAlgorithm

# Lossless gzip with byte shuffling (GZIP_2 = shuffled gzip; this is the default algorithm)
opts = CompressionOptions(algorithm=CompressionAlgorithm.GZIP_2)

coadd.writeFits(
    "coadd.fits",
    imageOptions=opts,
    maskOptions=opts,
    varianceOptions=opts,
)

In [ ]:
plt.figure(figsize=(10,10))
afwDisplay.setDefaultBackend("matplotlib")  # or "firefly" on the RSP
display = afwDisplay.Display(frame=1)
#display.scale("asinh", "zscale")
display.scale("asinh", -1, 25, Q=8)
display.mtv(coadd)
display.setMaskTransparency(0)

In [ ]:
plt.imshow(coadd.image.array[2000:2100, 2000:2100],norm='log')